# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alex-jb/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking Lane 2 because it turns straight into a decision a person makes every week — *which pages should an editor review first?* — instead of an abstract "predict decline." My Week-1 notebooks already showed the raw material is there: decline is common but uneven, and on the pipeline's **client-holdout** split a learned ranking beat a hand rule roughly **3×** at precision@50. Lane 2 also lets me build on the honest-validation habit I practiced in Notebook 02 (grouped train/test by `client_id`, precision@K) rather than starting cold. I'm holding the choice loosely — I can confirm or switch until the end of Week 4 — but the numbers in section 3 make it look worth the next 7 weeks.

In [1]:
import os, sys, pandas as pd, numpy as np

# Resolve the repo root whether we run at root, from work/notebooks/, or in Colab.
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    REPO = "flyrank-ml-internship"
    if not os.path.isdir(REPO):
        os.system(f"git clone --depth 1 https://github.com/alex-jb/{REPO} {REPO}")
    os.chdir(REPO)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.isdir("data/raw"), "could not locate data/raw"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")   # 30k pseudonymized pages, 32 clients
down = df["trend_direction"].str.lower().eq("down")
print(f"rows: {len(df):,} | clients: {df['client_id'].nunique()}")
print(f"decline base rate: {down.mean():.3f}  ({down.sum():,} pages trending down)")
# Backs the lane: decline is common -> you can't "refresh everything", you must prioritize.

rows: 30,000 | clients: 32
decline base rate: 0.542  (16,262 pages trending down)


## 2. The question: decision, action, cost of a wrong call

**The question.** *Given only safe, leakage-checked signals, which pages should a content team review first for refresh, expansion, protection, or pruning?*

- **Decision it improves:** where a content team spends its limited weekly refresh hours — which pages to look at *first*.
- **Unit of analysis:** one **page** (content item). Later time-window labels will be built per page, split by `client_id` so a client never appears in both train and test.
- **Output:** a **ranked queue** of pages, each with a suggested action (refresh / expand / protect / prune / monitor), reason codes, and a confidence label — never a black-box number.
- **Who acts, and what they do:** an SEO/content **editor** works the queue top-down and takes one action per page. If nobody would act differently, the work has no customer — here, they reorder their week.
- **Cost of a wrong call:** a **false positive** near the top burns scarce editor hours on a page that wouldn't have benefited (and risks disturbing a page that was fine); a **false negative** leaves a genuinely declining, high-demand page unattended while its traffic compounds downward. Editors only reach the *top* of the list, so errors near the top cost the most — which is why the metric is **precision@K**, not overall accuracy.
- **Why data/ML helps at all:** a plain if-rule ("stale AND visible") is a fine *baseline*, but decline here is driven by many tangled, shifting signals (impression-history depth, position, age, CTR, engagement). The pipeline's hand rule scored **0.24** precision@50 while a learned model reached **0.74**. The pattern is real but too messy to write by hand — which is exactly when ML earns its place, and not before.

In [2]:
visible = df["impressions_90d"] >= 100          # "visible" = actually shown to users in the last 90d
pool = df[down & visible]
print(f"weekly candidate pool (declining AND visible): {len(pool):,}  "
      f"({len(pool)/len(df)*100:.1f}% of all pages)")
print("An editor can't hand-review ~13k pages a week -> the job is RANKING, not flagging.")

weekly candidate pool (declining AND visible): 13,152  (43.8% of all pages)
An editor can't hand-review ~13k pages a week -> the job is RANKING, not flagging.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter CSV that make Lane 2 look worth 7 weeks: decline is **common** (so you must prioritize), the value at risk is **concentrated** (so a short ranked list captures most of it), and a hand rule is **beaten** by a learned model on honest validation (so ML earns its place).

In [3]:
# [1] decline is common but uneven -> "refresh everything" is not a strategy
print(f"[1] decline base rate: {down.mean():.1%}  ({down.sum():,}/{len(df):,} pages)")

# [2] at-risk traffic is concentrated -> ranking by impact pays off (precision@K is the right metric)
imp = pool["impressions_90d"].sort_values(ascending=False).to_numpy()
top10 = max(1, int(len(imp) * 0.10))
print(f"[2] top 10% of declining-visible pages hold {imp[:top10].sum()/imp.sum():.1%} "
      f"of all at-risk impressions -> a short ranked list captures most of the value")

# [3] the messy pattern beats a hand rule -> ML earns its place
#     (client-holdout numbers from the committed pipeline report, scripts/run_all.py)
print(f"[3] precision@50 (client-holdout): baseline rule 0.240 -> random forest 0.740 "
      f"(~{0.740/0.240:.1f}x)")

[1] decline base rate: 54.2%  (16,262/30,000 pages)
[2] top 10% of declining-visible pages hold 61.4% of all at-risk impressions -> a short ranked list captures most of the value
[3] precision@50 (client-holdout): baseline rule 0.240 -> random forest 0.740 (~3.1x)


## 4. Careful words: what I can and can't claim

**What I *can* say:** *observed* and *directional* associations on this pseudonymized sample, and a *decision-support* ranked queue with reason codes — "these pages look most worth a human's attention first," always reviewable by a person.

**What I will *never* say:** that I *proved a Google ranking factor*; that a refresh *will* recover a page (a declining page may be consolidating, seasonal, or just noise — correlation is not a guarantee); or that any score is a causal claim.

**A trap I'm naming now (backed by the code below):** the starter's `is_declining_label` is **defined** from `trend_direction` — they agree 100%. A model trained on it just relearns FlyRank's own rule, not the world. So `trend_direction` and `trend_pct` are **never** features, and for the capstone I'll reframe the target to an **observed** outcome measured in a **future** time window from the warehouse (a page's traffic in a later month), with a strict leakage audit. I'll also respect the data gotchas: rate columns are ×100 (`ctr = 0.76` means 0.76%), and `avg_position = 0` means "no data," not rank zero.

In [4]:
# Proof the starter label is DEFINED, not observed:
lab_from_rule = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"agreement of (trend_direction=='down') with itself as a label: "
      f"{(lab_from_rule == down.astype(int)).mean():.3f}  -> defined by a rule, not measured")
print("Fix for the capstone: an OBSERVED future-window label + leakage audit "
      "(never trend_direction / trend_pct as features).")

agreement of (trend_direction=='down') with itself as a label: 1.000  -> defined by a rule, not measured
Fix for the capstone: an OBSERVED future-window label + leakage audit (never trend_direction / trend_pct as features).


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.